In [1]:
from pathlib import Path

import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

from QuantNado import BamStore
from QuantNado.downstream import (
    annotate_intervals,
    extract_feature_ranges,
    extract_metadata,
    extract_promoters,
    feature_counts,
    load_gtf,
    plot_pca_scatter,
    plot_pca_scree,
    reduce_byranges_signal,
    run_pca,
)


In [2]:
fig_dir = Path("./figures")
fig_dir.mkdir(exist_ok=True)

# Load Dataset

In [3]:
ds = BamStore.open("dataset", backend="zarr")
ds

2025-12-20 00:50:00.532 | INFO     | QuantNado.bam_store:open:372 - Opening zarr store at: dataset.zarr


<xarray.Dataset> Size: 4GB
Dimensions:        (sample: 8, position_flat: 154755866, contig: 3)
Coordinates:
  * sample         (sample) int64 64B 0 1 2 3 4 5 6 7
  * position_flat  (position_flat) int64 1GB 0 1 2 ... 154755864 154755865
  * contig         (contig) int64 24B 0 1 2
    contig_offset  (contig) int64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    contig_length  (contig) int64 24B dask.array<chunksize=(3,), meta=np.ndarray>
Data variables:
    signal         (sample, position_flat) uint16 2GB dask.array<chunksize=(1, 64000), meta=np.ndarray>
Attributes: (12/21)
    assay_by_sample:         ['ATAC', 'ATAC', 'ChIP', 'ChIP', 'ChIP', 'ChIP',...
    metadata_control:        ['', '', 'Input', 'Input', 'Input', 'Input', '',...
    metadata_ip:             ['', '', 'MLL', 'MLL', 'MLL', 'MLL', '', '']
    metadata_replicate:      ['', '', '', '', '', '', 'rep1', 'rep1']
    metadata_scaling_group:  ['default', 'default', 'default', 'default', 'de...
    metadata_timepoint:      ['24hr', '24hr', '24hr', '24hr', '24hr', '24hr',...
    ...                      ...
    assays:                  ATAC,ChIP,RNA
    sample_names:            ['SEM-DMSO', 'SEM-MENi', 'SEM-DMSO-MLL_Input', '...
    contig_names:            ['chr21', 'chr22', 'chrY']
    structure:               ragged (sample × position_flat with contig offsets)
    bin_size:                1
    average_sparsity:        85.64%

# Explore Zarr Dataset



In [4]:
# Check dimensions and coordinates
print("Dimensions:", ds.dims)
print("\nCoordinates:", list(ds.coords))
print("\nAttributes:", ds.attrs)
print("\nData variables:", list(ds.data_vars))

Dimensions: FrozenMappingWarningOnValuesAccess({'sample': 8, 'position_flat': 154755866, 'contig': 3})

Coordinates: ['contig_offset', 'sample', 'contig', 'contig_length', 'position_flat']

Attributes: {'assay_by_sample': ['ATAC', 'ATAC', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'RNA', 'RNA'], 'metadata_control': ['', '', 'Input', 'Input', 'Input', 'Input', '', ''], 'metadata_ip': ['', '', 'MLL', 'MLL', 'MLL', 'MLL', '', ''], 'metadata_replicate': ['', '', '', '', '', '', 'rep1', 'rep1'], 'metadata_scaling_group': ['default', 'default', 'default', 'default', 'default', 'default', 'default', 'default'], 'metadata_timepoint': ['24hr', '24hr', '24hr', '24hr', '24hr', '24hr', '24hr', '24hr'], 'metadata_treatment': ['DMSO', 'MENi', 'DMSO', 'DMSO', 'MENi', 'MENi', 'DMSO', 'MENi'], 'metadata_r1': ['/ceph/project/milne_group/cchahrou/project/2025-12-17_menin_inh_24hr/fastqs/atac/SEM-DMSO_R1.fastq.gz', '/ceph/project/milne_group/cchahrou/project/2025-12-17_menin_inh_24hr/fastqs/atac/SEM-MENi_R1.fastq.gz

In [5]:
# Helpers for ragged coordinates and metadata
metadata_df = extract_metadata(ds)

# Reduce by BED file

In [6]:
# promoters = "/Users/catherine/work/project/QuantNado/data/hg38/promoters_1024bp.bed"

# promoter_ds = reduce_byranges_signal(ds["signal"], bed_file=promoters, reduction="mean")
# promoter_ds

## PCA analysis

Perform Principal Component Analysis on the promoter dataset to visualize sample relationships:

In [7]:
# # Ensure numeric dtype and explicit chunking to avoid auto on object dtypes
# promoter_mean = promoter_ds["mean"].astype("float32")
# promoter_signal = promoter_mean.chunk({"ranges": 10000})

# pca_object, pca_result = run_pca(
#     input_array=promoter_signal,
#     n_components=2,
#     nan_handling_strategy="drop",
#     standardize=True,
#     random_state=42,
#     subset_size=1_000,
#     subset_strategy="random",
#     svd_solver="randomized",
# )

# plot_pca_scree(
#     pca_object=pca_object,
#     filepath=f"{fig_dir}/pca_scree.png",
# )

# plot_pca_scatter(
#     pca_object,
#     pca_result,
#     xaxis_pc=1,
#     yaxis_pc=2,
#     metadata_df=metadata_df,
#     colour_by="assay",
#     shape_by="treatment",
#     sample_column="sample_id",
#     filepath=f"{fig_dir}/pca_plot.png",
# )

# Extract Feature counts

In [ ]:
gtf_file = "/Users/catherine/work/project/QuantNado/data/hg38/hg38.ncbiRefSeq.gtf"


counts_df, feature_meta = feature_counts(
    ds["signal"],
    gtf_file=gtf_file,
    feature_type="exon",
    aggregate_by="gene_id",
    metadata_df=metadata_df,
    sample_col="sample_id",
    integerize=True,
 )


counts_df = counts_df.fillna(0).astype("int64")


display(counts_df.head())
display(feature_meta.head())

,0,1,2,3,4,5,6,7
gene_id,,,,,,,,
A1BG,0,0,0,0,0,0,0,0
A1BG-AS1,0,0,0,0,0,0,0,0
A1CF,0,0,0,0,0,0,0,0
A2M,0,0,0,0,0,0,0,0
A2M-AS1,0,0,0,0,0,0,0,0


,gene_id,contig,start,end,range_length
0,A1BG,chr19,58345183,58353492,3374
1,A1BG-AS1,chr19,58351970,58355183,2126
2,A1CF,chr10,50799409,50885627,129495
3,A2M,chr12,9067708,9116229,23256
4,A2M-AS1,chr12,9065177,9068055,6762


In [11]:
# Align metadata to counts columns (samples) robustly
counts_df = counts_df.copy()
# remove rows with 0 in all samples
counts_df = counts_df.loc[~(counts_df == 0).all(axis=1)]
counts_df.columns = counts_df.columns.astype(str)
counts_df

,0,1,2,3,4,5,6,7
gene_id,,,,,,,,


In [10]:
# Align metadata to counts columns (samples) robustly
counts_df = counts_df.copy()
# remove rows with 0 in all samples
counts_df = counts_df.loc[~(counts_df == 0).all(axis=1)]
counts_df.columns = counts_df.columns.astype(str)


candidates = [
    c for c in ["sample_id", "sample", "sample_name"] if c in metadata_df.columns
]
if candidates:
    sample_col = candidates[0]
    metadata_idx = metadata_df.copy()
    metadata_idx[sample_col] = metadata_idx[sample_col].astype(str)
    metadata_idx = metadata_idx.set_index(sample_col)
else:
    metadata_idx = metadata_df.copy()
    metadata_idx.index = metadata_idx.index.astype(str)


common_samples = counts_df.columns.intersection(metadata_idx.index)
if len(common_samples) == 0:
    raise ValueError("No overlapping samples between counts columns and metadata")


counts_df = counts_df.loc[:, common_samples]
metadata_for_deseq = metadata_idx.loc[common_samples]


dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata_for_deseq,
    design="~treatment",
)


dds.fit_size_factors()
dds.fit_genewise_dispersions()


dds.obs["size_factors"]
dds.var["genewise_dispersions"]

ValueError: No overlapping samples between counts columns and metadata

In [ ]:
dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata_for_deseq,
    design="~treatment",
)


dds.fit_size_factors()


dds.obs["size_factors"]
dds.fit_genewise_dispersions()


dds.var["genewise_dispersions"]